# Data Ingestion from Cloud Storage

## A. Data Ingestion Patterns From Cloud Object Storage

Data ingestion is a critical component of modern Lakehouse architecture, enabling organizations to take advantage of large volumes of data stored in cloud object storage systems.

```text
+------------------+      +---------------------+      +---------------+
|  Cloud Storage   | ---> |   Data Ingestion    | ---> |  Delta Table  |
|                  |      |                     |      |               |
| • CSV            |      | • CREATE TABLE AS   |      |     Delta     |
| • JSON           |      | • COPY INTO         |      |     Table     |
| • Parquet        |      | • AUTO LOADER       |      |               |
| • etc.           |      |                     |      |               |
+------------------+      +---------------------+      +---------------+
```

### CREATE TABLE AS (CTAS)
```sql
CREATE TABLE new_table AS
          SELECT *
          FROM read_files(
            <path_to_file(s)>,
            format => '<file_type>',
            <other_format_specific_options>
          );
```

- `CREATE TABLE AS (CTAS)` creates a Delta table by default from files in cloud object storage.
- The `read_files()` function reads files under a provided location and returns the data in tabular form.

### COPY INTO
```sql
CREATE TABLE new_table;
        
        COPY INTO new_table
        FROM '<dir_path>'
        FILEFORMAT = <file_type>
        FORMAT_OPTIONS (<options>)
        COPY_OPTIONS (<options>)
```
- Copy files from cloud storage into the Delta table; this performs a bulk load from files in cloud object storage into the table.
- skips any files that have already been loaded into the table, and only new files will be ingested.

### Incremental Batch or Streaming (`AUTO LOADED`)
```python
(spark
  .readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "<checkpoint_path>")
    .load("/Volumes/catalog/schema/files")
  .writeStream
    .option("checkpointLocation", "<checkpoint_path>")
    .trigger(processingTime="5 seconds")
    .toTable("catalog.database.table")
)
```

```sql
CREATE OR REFRESH STREAMING TABLE
  catalog.schema.table
SCHEDULE EVERY 1 HOUR
AS
SELECT *
FROM STREAM read_files(
  '<dir_path>',
  format => '<file_type>'
)
```

#### Read volumes with python via CTAS
To ingest like CTAS with python you can of course read like a human being WRITING A PYSPARK!!!

```python
# Read the volume (unparsed files)
df = spark.read.format(
    "parquet"
).load("volume-path")

# Write to delta table
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{my_catalog}.bronze.table_name")
```

#### Reading Volumes with `COPY INTO`
with `SQL`

- Since you are not overwriting your data a table must already exist, thus when reading files there can be a schema mismatch, especially when you did not enforce it like in pyspark

```sql
DROP TABLE IF EXISTS bronze_tabel;


-- We added a schema
CREATE TABLE bronze_table(
    user_id STRING,
    some_int BIGINT
);

COPY INTO bronze_table
    FROM "volume_path"  --if this has 3 columns we will have a problem here
    FILEFORMAT = "json";
```



use:
```sql
FILEFORMAT = "json"
COPY_OPTIONS(
    "merge_schema" = "True"
)
```

instead